# Datenanalyse: Mikroplastik-Verschmutzung in Japan (2014–2023)

Dieses Jupyter Notebook wertet die Mikroplastik-Dichte in 7 Meeresregionen Japans aus (Quelle: AOMI / Umweltministerium Japan).

### Enthaltene Analysen & Visualisierungen:
1. **Datenimport & Prüfung** der aggregierten Regionaldaten
2. **Referenz 1 (Basisjahr 2014):** Prozentuale Veränderung vom Startjahr bis 2023
3. **Referenz 2 (Japan-Median & Fokus-Regionen):** Hervorhebung wichtiger Hotspots gegenüber dem landesweiten Median
4. **Heatmap (Wärmekarte):** Gesamtübersicht über alle Jahre und Regionen

## 1. Bibliotheken laden

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LogNorm
from IPython.display import display

# >>> 1. HIER PFAD ZU DEINER AOMI-CSV ANPASSEN <<<
BASE = Path("/Users/jennyjanac/Desktop/Life/Muros_2026_japan_hsd/Data_sets/data_muros/data/data_download/survey_data.csv") 
CSV_DATEI = BASE / "deine_aomi_daten.csv"  # <-- Hier den genauen Dateinamen eintragen

# Output-Ordner erstellen (speichert alle Grafiken und Tabellen automatisch)
OUT = Path("output")
OUT.mkdir(exist_ok=True)

print("CSV-Datei gefunden:", CSV_DATEI.exists())

# >>> 2. CSV EINLESEN & INSPEKTION <<<
if CSV_DATEI.exists():
    # Falls die CSV Semikolons statt Kommas nutzt, sep=";" ergänzen
    df_raw = pd.read_csv(CSV_DATEI)
    print("Anzahl Zeilen & Spalten:", df_raw.shape)
    print("Spaltennamen:", list(df_raw.columns))
    display(df_raw.head(5))

ModuleNotFoundError: No module named 'seaborn'

## 2. Daten einlesen
Hier sind die bereits nach den 7 Meeresregionen zusammengefassten Daten (2014–2023) hinterlegt.

In [ ]:
daten = {
    "Year": [2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023],
    "1_SeaOfJapan_North": [4.162, 3.704, 6.239, 0.786, 2.052, 5.744, 1.060, 14.151, 10.502, 0.537],
    "2_SeaOfJapan_West":  [35.476, 0.840, 1.642, 1.010, 13.282, 2.312, 1.812, 1.311, 0.811, 0.384],
    "3_EastChinaSea":     [1.461, 0.997, 0.534, 0.503, 0.693, 1.367, 0.594, 0.515, 0.435, 0.650],
    "4_Pacific_South":    [26.114, 1.100, 0.173, 1.102, 5.264, 1.890, 0.395, 0.474, 0.554, 0.442],
    "5_Pacific_Central":  [0.472, 1.598, 1.058, 1.100, 3.495, 1.546, 0.628, 0.070, 0.222, 0.222],
    "6_Pacific_North":    [1.583, 11.393, 12.505, 0.851, 3.121, 3.489, 0.641, 0.465, 0.289, 0.151],
    "7_Ogasawara":        [0.056, 0.056, 0.056, 0.096, 0.321, 0.435, 0.076, 0.003, 0.003, 0.003]
}

df = pd.DataFrame(daten).set_index("Year")
display(df)

## 3. Referenz 1: Prozentuale Veränderung im Vergleich zu 2014
Um die unterschiedlichen Ausgangsniveaus vergleichbar zu machen, berechnen wir die relative Veränderung vom Basisjahr 2014 bis zum Jahr 2023.

In [ ]:
# Prozentuale Veränderung relativ zu 2014 berechnen
df_relativ = ((df - df.loc[2014]) / df.loc[2014]) * 100
veraenderung_gesamt = df_relativ.loc[2023].sort_values()

plt.figure(figsize=(10, 5))
farben = ["#2a9d8f" if x < 0 else "#e76f51" for x in veraenderung_gesamt]
bars = plt.barh(veraenderung_gesamt.index, veraenderung_gesamt.values, color=farben)

plt.axvline(0, color="black", linewidth=1)
plt.title("Veränderung der Mikroplastik-Dichte: 2023 im Vergleich zu 2014 (in %)", fontsize=13, fontweight="bold")
plt.xlabel("Veränderung gegenüber 2014 (%)")
plt.xlim(-115, 10)

for bar in bars:
    wert = bar.get_width()
    plt.text(wert - 2, bar.get_y() + bar.get_height()/2,
             f"{wert:.1f} %", va="center", ha="right", fontweight="bold")

plt.tight_layout()
plt.show()

## 4. Referenz 2: Fokus-Regionen vs. Japan-Median
Statt alle 7 Linien gleichzeitig bunt darzustellen (Spaghetti-Effekt), heben wir nur die wichtigsten Regionen hervor und nutzen den **japanweiten Median** als Referenzlinie.

In [ ]:
japan_median = df.median(axis=1)

fokus_regionen = {
    "1_SeaOfJapan_North": ("#d62828", "Hotspot: Japanisches Meer (Nord)"),
    "2_SeaOfJapan_West":  ("#f77f00", "Japanisches Meer (West)"),
    "7_Ogasawara":        ("#0077b6", "Referenz (Ozean): Ogasawara")
}

plt.figure(figsize=(11, 6))

# Hintergrundlinien in Hellgrau
for spalte in df.columns:
    if spalte not in fokus_regionen:
        plt.plot(df.index, df[spalte], color="lightgrey", linewidth=1.5, alpha=0.8)

# Japan-Median als Referenz
plt.plot(df.index, japan_median, color="black", linestyle="--", linewidth=2.5, label="Referenz: Japan-Median (alle Zonen)")

# Fokus-Regionen hervorheben
for spalte, (farbe, label_text) in fokus_regionen.items():
    plt.plot(df.index, df[spalte], color=farbe, marker="o", linewidth=2.8, label=label_text)

plt.yscale("log")
plt.title("Fokus-Regionen im Vergleich zum japanweiten Median (2014–2023)", fontsize=13, fontweight="bold")
plt.ylabel("Mülldichte (log. Skala)")
plt.xlabel("Jahr")
plt.xticks(df.index)
plt.legend(loc="lower left", frameon=True)
plt.tight_layout()
plt.show()

## 5. Heatmap (Wärmekarte) für den schnellen Gesamtüberblick
Hier werden Zahlen in Farben übersetzt. Dadurch sind Hotspots und Rückgänge auf einen Blick erkennbar.

In [ ]:
plt.figure(figsize=(11, 5))
sns.heatmap(
    df.T,
    annot=True,
    fmt=".2f",
    cmap="YlOrRd",
    norm=LogNorm(),
    cbar_kws={"label": "Mülldichte (log. Skala)"}
)

plt.title("Wärmekarte (Heatmap) der Mikroplastik-Dichte nach Region und Jahr", fontsize=13, fontweight="bold")
plt.xlabel("Jahr")
plt.ylabel("Meeresregion")
plt.tight_layout()
plt.show()